# DevFlow - Lab: Triagem e Planejamento com LLM

**Curso:** Agentic Engineering - SkillGo
**Prof.:** Ives Santos

---

Este notebook isola as **duas etapas do DevFlow que chamam o LLM para produzir trabalho**:

```
issue + contexto --> [TRIAGEM] --> Triagem --> [PLANEJAMENTO] --> Plano
```

| Etapa | Recebe | Devolve |
|---|---|---|
| Triagem | issue + trechos da base de conhecimento | tipo, severidade, prioridade, esforco, componentes |
| Planejamento | issue + trechos + **a triagem** (+ feedback, se houver) | passos, testes, riscos |

Sem grafo, sem RAG, sem guardrails: o contexto que o RAG recuperaria ja vem pronto numa celula.
O foco e o que acontece **dentro** de cada etapa - contrato, prompt e chamada ao modelo.

> **Ideia central:** cada etapa e sempre a mesma receita:
> **contrato** (o que deve sair) + **prompt** (o que o modelo sabe) + **`with_structured_output`** (a chamada).

## 1. Ambiente

Voce precisa de uma chave da Anthropic (console.anthropic.com).

**No Colab:** clique no icone de chave (Secrets) na barra lateral, crie o segredo
`ANTHROPIC_API_KEY` e ative o acesso do notebook. Se preferir, a celula pede a chave
digitada - ela nao fica salva no notebook.

In [ ]:
!pip install -q "langchain-anthropic>=1.0" "pydantic>=2.7"

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    if not os.getenv("ANTHROPIC_API_KEY"):
        from getpass import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")

assert os.environ["ANTHROPIC_API_KEY"].startswith("sk-ant-"), "chave ausente ou em formato inesperado"
print("chave carregada:", len(os.environ["ANTHROPIC_API_KEY"]), "caracteres")

---
## 2. Contratos

O formato que cada etapa **deve** devolver:

- `Issue` - a entrada; `criterios_numerados()` cria `CA1`, `CA2`... para o modelo citar;
- `Triagem` - saida da etapa 1;
- `PassoPlano` / `Plano` - saida da etapa 2.

**O contrato tambem e a instrucao.** Nao existe no prompt uma frase dizendo "escreva um resumo,
depois os passos". Quem diz ao modelo o que cada campo significa e o proprio contrato:

| No Python | Vira, para o modelo |
|---|---|
| nome da classe | nome da ferramenta que ele deve chamar |
| docstring da classe | descricao da ferramenta |
| nome e tipo do campo | nome e tipo do argumento |
| campo sem valor padrao | argumento obrigatorio |
| `min_length=1` | `minItems: 1` |
| `Literal[...]` | `enum` - so esses valores |
| `Field(description=...)` | **a explicacao do campo**, lida pelo modelo |

Sem `description`, o modelo **deduz** o campo pelo nome. Com ela, voce **especifica**: o que e,
formato e tamanho. Por isso todos os campos que o LLM preenche abaixo tem descricao.

A `Issue` nao tem descricoes porque o LLM nao a preenche - ela so entra no prompt como texto.

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field


class Issue(BaseModel):
    id: str
    projeto: str
    autor: str
    criada_em: str
    labels: List[str] = Field(default_factory=list)
    titulo: str = Field(min_length=5)
    descricao: str = Field(min_length=20)
    criterios_aceite: List[str] = Field(min_length=1)

    def criterios_numerados(self) -> List[str]:
        return [f"CA{i}: {c}" for i, c in enumerate(self.criterios_aceite, start=1)]

    def ids_criterios(self) -> List[str]:
        return [f"CA{i}" for i in range(1, len(self.criterios_aceite) + 1)]


class Triagem(BaseModel):
    """Classificacao da issue segundo a politica de engenharia."""

    tipo: Literal["bug", "feature", "debito_tecnico", "documentacao", "suporte"] = Field(
        description="Natureza da issue: bug = comportamento incorreto de algo que ja existe; "
                    "feature = capacidade nova; debito_tecnico = melhoria interna sem mudanca visivel; "
                    "documentacao = so texto; suporte = duvida ou pedido operacional"
    )
    severidade: Literal["baixa", "media", "alta", "critica"] = Field(
        description="Gravidade do impacto, conforme a classificacao de severidade da politica no contexto"
    )
    prioridade: Literal["P0", "P1", "P2", "P3"] = Field(
        description="Urgencia de atendimento; deve seguir a correspondencia severidade -> prioridade da politica"
    )
    componentes: List[str] = Field(
        description="Servicos ou modulos afetados, usando os nomes que aparecem no contexto, ex.: pricing-service"
    )
    esforco: Literal["XS", "S", "M", "L", "XL"] = Field(
        description="Tamanho estimado do trabalho, conforme a escala de esforco da politica"
    )
    justificativa: str = Field(
        description="Por que essa classificacao, em ate 5 frases, citando a regra da politica que a sustenta"
    )
    fontes: List[str] = Field(
        default_factory=list,
        description="IDs dos trechos do contexto usados na classificacao, ex.: politica-de-engenharia#2",
    )


class PassoPlano(BaseModel):
    """Uma etapa do plano, executavel por uma pessoa desenvolvedora."""

    ordem: int = Field(description="Posicao do passo na sequencia de execucao, comecando em 1")
    titulo: str = Field(description="Acao do passo em ate 8 palavras, comecando por verbo")
    detalhe: str = Field(description="O que fazer e onde (arquivo, modulo ou servico), em 1 ou 2 frases")
    criterios_atendidos: List[str] = Field(
        default_factory=list,
        description="IDs dos criterios de aceite cobertos por este passo, ex.: ['CA1','CA3']",
    )


class Plano(BaseModel):
    """Plano tecnico de resolucao da issue."""

    resumo: str = Field(description="A estrategia da correcao em no maximo 2 frases")
    passos: List[PassoPlano] = Field(min_length=1, description="Etapas em ordem de execucao")
    testes: List[str] = Field(
        min_length=1,
        description="Cenarios de teste automatizado a escrever, um por item, com entrada e resultado esperado",
    )
    riscos: List[str] = Field(
        default_factory=list,
        description="O que pode dar errado ao aplicar o plano e dependencias ainda nao confirmadas",
    )
    fontes: List[str] = Field(
        default_factory=list,
        description="IDs dos trechos do contexto usados no plano, ex.: arquitetura-checkout#3",
    )

### 2.1 O que o modelo realmente recebe

`convert_to_anthropic_tool` e a conversao que o `with_structured_output` faz por baixo dos panos.
A saida abaixo e a definicao de ferramenta enviada ao Claude junto com o prompt - repare que
cada `description` escrita acima esta la, colada no seu campo.

In [ ]:
import json
from langchain_anthropic.chat_models import convert_to_anthropic_tool

ferramenta = convert_to_anthropic_tool(Plano)

print("ferramenta:", ferramenta["name"], "-", ferramenta["description"])
print("obrigatorios:", ferramenta["input_schema"]["required"])
print()
for nome, campo in ferramenta["input_schema"]["properties"].items():
    print(f"{nome:<8} ({campo['type']}) {campo.get('description', '')}")

In [ ]:
# A definicao completa, incluindo o PassoPlano aninhado dentro de "passos"
print(json.dumps(ferramenta, indent=2, ensure_ascii=False))

---
## 3. Entradas: a issue e o contexto

A issue e a ISSUE-1042 do projeto. O **contexto** sao os trechos que o RAG do DevFlow
recuperaria para ela - aqui escritos a mao, para o notebook nao depender de busca.

Cada trecho tem um **ID** (`documento#secao`). O modelo sera obrigado a citar esses IDs em
`fontes`, e isso permite conferir depois se ele citou algo que realmente recebeu.

In [ ]:
issue = Issue(
    id="ISSUE-1042",
    projeto="loja-aurora",
    autor="marina.souza",
    criada_em="2026-08-14",
    labels=["checkout", "pricing", "cliente-impactado"],
    titulo="Cupom de desconto e aplicado duas vezes quando o cliente volta para a etapa de pagamento",
    descricao=(
        "Clientes relataram que o valor final do pedido fica menor do que deveria. Reproduzimos assim: "
        "adicionar dois itens ao carrinho, avancar ate PAGAMENTO, aplicar o cupom PRIMEIRA10, voltar para "
        "ENDERECO e avancar de novo para PAGAMENTO. Na segunda passagem o desconto de 10% aparece somado "
        "duas vezes e o total sai 20% menor. O suporte registrou 37 pedidos afetados em 48 horas. Nos logs "
        "aparece uma segunda chamada de POST /pricing/quote sem cabecalho de idempotencia logo apos o "
        "evento cart.state_changed."
    ),
    criterios_aceite=[
        "Aplicar o mesmo codigo de cupom mais de uma vez na mesma cotacao nao deve alterar o total alem do primeiro desconto",
        "Voltar e avancar entre as etapas ENDERECO e PAGAMENTO deve manter o total do pedido estavel",
        "Deve existir teste automatizado que reproduza o cenario do defeito e falhe antes da correcao",
        "A correcao deve poder ser desligada sem novo deploy",
    ],
)

contexto = [
    {"id": "politica-de-engenharia#2", "titulo": "Classificacao de severidade",
     "texto": "critica: perda financeira direta, indisponibilidade total ou vazamento de dados. "
              "alta: erro de calculo visivel ao cliente, degradacao relevante ou bloqueio de fluxo principal. "
              "media: comportamento incorreto com contorno possivel. baixa: ajuste cosmetico. "
              "Prioridade P0 para severidade critica em producao; P1 para alta; P2 para media; P3 para baixa."},
    {"id": "politica-de-engenharia#3", "titulo": "Estimativa de esforco",
     "texto": "XS (ate 2 horas), S (ate 1 dia), M (ate 3 dias), L (ate 1 semana), XL (acima de uma semana)."},
    {"id": "politica-de-engenharia#4", "titulo": "Mudancas em regras de preco",
     "texto": "Alteracao no pricing-service que afete valores cobrados exige teste de regressao, revisao de "
              "uma pessoa do time de Pagamentos e liberacao gradual por feature flag pricing.<nome_da_regra>."},
    {"id": "arquitetura-checkout#3", "titulo": "Aplicacao de cupons",
     "texto": "Cupons sao aplicados no pricing-service, modulo pricing/discounts/coupon.py, funcao aplicar_cupom. "
              "Um mesmo codigo de cupom so pode incidir uma vez por cotacao. Cupons aplicados ficam na tabela "
              "cart_coupons, chave unica (cart_id, coupon_code)."},
    {"id": "arquitetura-checkout#4", "titulo": "Idempotencia de cotacao",
     "texto": "POST /pricing/quote deve ser idempotente por cart_id. O cabecalho Idempotency-Key e obrigatorio "
              "desde a versao 3.4 do checkout-api; chamadas sem a chave sao aceitas por retrocompatibilidade."},
    {"id": "padroes-de-testes#2", "titulo": "Testes de regressao de preco",
     "texto": "Bugs de calculo exigem teste que reproduza o defeito e falhe antes da correcao, em "
              "tests/unit/pricing/test_coupon_regression.py, com o id da issue no nome do teste."},
]

print(issue.id, "|", len(issue.criterios_aceite), "criterios |", len(contexto), "trechos de contexto")

---
## 4. Prompts

Cada etapa tem **dois** textos:

| Texto | Papel | Muda entre issues? |
|---|---|---|
| mensagem de **sistema** | quem o modelo e, regras que nunca mudam | nao |
| mensagem de **usuario** | a issue, o contexto e o pedido | sim |

Um cuidado aparece nos dois prompts: **citacao obrigatoria**. O modelo precisa listar em
`fontes` os IDs dos trechos que usou e, no plano, ligar cada passo a um `CA`. E isso que
permite conferir a resposta depois, com codigo.

In [ ]:
REGRA_CONCISAO = (
    "Seja direto e especifico. Justificativas com no maximo cinco frases; passos e itens "
    "de lista com uma ou duas frases cada. Nao repita o enunciado da issue."
)

SISTEMA_TRIAGEM = f"""Voce e o DevFlow, um agente de triagem de issues de engenharia de software.
Classifique a issue usando exclusivamente a politica interna fornecida no contexto.
Escolha UMA classificacao e defenda essa; nao apresente alternativas condicionais.
Nao invente servicos, arquivos ou incidentes que nao aparecam no contexto.
Em `fontes`, liste os IDs dos trechos do contexto que sustentam a sua classificacao.
{REGRA_CONCISAO}"""

SISTEMA_PLANO = f"""Voce e o DevFlow, um agente que transforma uma issue triada em plano tecnico.
O plano deve ser executavel por outra pessoa desenvolvedora sem contexto adicional.
Cada criterio de aceite (CA1, CA2, ...) precisa aparecer em `criterios_atendidos` de
pelo menos um passo - essa cobertura e verificada automaticamente.
Baseie decisoes tecnicas no contexto fornecido e cite os IDs usados em `fontes`.
Ao propor nome de arquivo, de flag ou de metrica que nao esteja no contexto, marque
explicitamente como sugestao a validar, e registre a dependencia em `riscos`.
{REGRA_CONCISAO}"""

### 4.1 Montando a mensagem de usuario

Funcoes pequenas que transformam objetos em texto. Repare na diferenca entre as duas:

- `prompt_triagem` envia **issue + contexto**;
- `prompt_plano` envia **issue + contexto + a triagem ja feita** e, se existir, um
  **feedback** a incorporar. E assim que a saida da etapa 1 vira entrada da etapa 2 -
  e que o plano pode ser refeito depois de uma revisao.

In [ ]:
def formatar_issue(issue: Issue) -> str:
    return (
        "ISSUE\n"
        f"id: {issue.id}\nprojeto: {issue.projeto}\nlabels: {', '.join(issue.labels)}\n"
        f"titulo: {issue.titulo}\n\ndescricao:\n{issue.descricao}\n\n"
        "criterios_aceite:\n" + "\n".join(f"- {c}" for c in issue.criterios_numerados())
    )


def formatar_contexto(contexto: list) -> str:
    blocos = [f"[{t['id']}] ({t['titulo']})\n{t['texto']}" for t in contexto]
    return "CONTEXTO DA BASE DE CONHECIMENTO\n" + "\n\n".join(blocos)


def prompt_triagem(issue: Issue, contexto: list) -> str:
    return f"{formatar_issue(issue)}\n\n{formatar_contexto(contexto)}\n\nClassifique a issue acima."


def prompt_plano(issue: Issue, contexto: list, triagem: Triagem, feedback: str | None = None) -> str:
    partes = [
        formatar_issue(issue),
        formatar_contexto(contexto),
        f"TRIAGEM JA FEITA\n{triagem.model_dump_json(indent=2)}",
    ]
    if feedback:
        partes.append(
            f"FEEDBACK A INCORPORAR\n{feedback}\n"
            "Este e um replanejamento: corrija especificamente os pontos acima."
        )
    partes.append("Produza o plano tecnico. IDs de criterio disponiveis: " + ", ".join(issue.ids_criterios()))
    return "\n\n".join(partes)


# Veja exatamente o que o modelo vai receber na triagem
print(prompt_triagem(issue, contexto)[:1200], "\n[...]")

---
## 5. O motor: uma funcao para chamar o LLM

Toda chamada do DevFlow passa por aqui. O que faz o trabalho pesado e
`with_structured_output(schema)`:

1. transforma o contrato Pydantic em JSON Schema e o declara como uma ferramenta;
2. obriga o modelo a responder **chamando essa ferramenta**;
3. valida a resposta e devolve uma **instancia** do contrato - nao texto.

Por isso nao existe parsing de texto em lugar nenhum deste notebook.

| Modelo | Quando usar |
|---|---|
| `claude-opus-5` | padrao do projeto, maior qualidade |
| `claude-sonnet-5` | bom equilibrio entre custo e qualidade |
| `claude-haiku-4-5` | mais barato e rapido, bom para experimentar |

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatAnthropic(model="claude-opus-5", max_tokens=8000)


def chamar(schema, sistema: str, usuario: str):
    estruturado = llm.with_structured_output(schema)
    return estruturado.invoke([SystemMessage(content=sistema), HumanMessage(content=usuario)])

---
## 6. Etapa 1: Triagem

Uma linha: contrato `Triagem` + prompt de sistema + prompt da issue.
O resultado ja e um objeto `Triagem` - `triagem.severidade` so pode ser um dos quatro valores.

In [ ]:
def triar(issue: Issue, contexto: list) -> Triagem:
    return chamar(Triagem, SISTEMA_TRIAGEM, prompt_triagem(issue, contexto))


triagem = triar(issue, contexto)

print("tipo        :", triagem.tipo)
print("severidade  :", triagem.severidade)
print("prioridade  :", triagem.prioridade)
print("esforco     :", triagem.esforco)
print("componentes :", triagem.componentes)
print("fontes      :", triagem.fontes)
print()
print("justificativa:", triagem.justificativa)

### 6.1 Conferindo a triagem sem LLM

O contrato garante o **formato**. Duas perguntas que ele nao responde - e que codigo simples responde:

- as `fontes` citadas **existem** no contexto enviado? (se nao, o modelo inventou a referencia)
- a prioridade e **coerente** com a severidade, segundo a politica?

In [ ]:
ids_contexto = {t["id"] for t in contexto}
inventadas = [f for f in triagem.fontes if f not in ids_contexto]

PRIORIDADE_ESPERADA = {"critica": "P0", "alta": "P1", "media": "P2", "baixa": "P3"}
esperada = PRIORIDADE_ESPERADA[triagem.severidade]

print("fontes inventadas:", inventadas or "nenhuma")
print("prioridade       :", "coerente" if triagem.prioridade == esperada else f"esperado {esperada}, veio {triagem.prioridade}")

---
## 7. Etapa 2: Planejamento

Mesma receita, contrato diferente. A novidade e a **triagem entrando no prompt**: o plano
nao reclassifica a issue, ele parte da classificacao ja feita.

In [ ]:
def planejar(issue: Issue, contexto: list, triagem: Triagem, feedback: str | None = None) -> Plano:
    return chamar(Plano, SISTEMA_PLANO, prompt_plano(issue, contexto, triagem, feedback))


plano = planejar(issue, contexto, triagem)

print("RESUMO:", plano.resumo)
print()
for p in plano.passos:
    print(f"{p.ordem}. {p.titulo}  {p.criterios_atendidos}")
    print(f"   {p.detalhe}")
print()
print("TESTES:")
for t in plano.testes:
    print(" -", t)
print("RISCOS:")
for r in plano.riscos:
    print(" -", r)
print()
print("fontes:", plano.fontes)

### 7.1 Conferindo a cobertura dos criterios

O prompt **pediu** que todo criterio aparecesse em algum passo. Pedir nao e garantir -
entao conferimos com uma operacao de conjuntos. No DevFlow, esta conta fica no guardrail de
saida, e um criterio descoberto devolve o fluxo para o planejamento.

In [ ]:
esperados = set(issue.ids_criterios())
cobertos = {ca for p in plano.passos for ca in p.criterios_atendidos}

print("esperados:", sorted(esperados))
print("cobertos :", sorted(cobertos & esperados))
print("faltando :", sorted(esperados - cobertos) or "nenhum")
print("fontes inventadas no plano:", [f for f in plano.fontes if f not in ids_contexto] or "nenhuma")

---
## 8. Replanejamento com feedback

No DevFlow o plano pode voltar para esta etapa - pelo revisor automatico ou pela pessoa
revisora - sempre com um **feedback**. A funcao e a mesma; so o prompt ganha o bloco
`FEEDBACK A INCORPORAR`.

Aqui simulamos o pedido de uma pessoa revisora.

In [ ]:
feedback = "Detalhar o plano de rollback: como desligar a correcao pela feature flag e o que monitorar apos ligar."

plano_v2 = planejar(issue, contexto, triagem, feedback=feedback)

print("RESUMO v2:", plano_v2.resumo)
print()
for p in plano_v2.passos:
    print(f"{p.ordem}. {p.titulo}  {p.criterios_atendidos}")
print()
print(f"passos: {len(plano.passos)} -> {len(plano_v2.passos)} | riscos: {len(plano.riscos)} -> {len(plano_v2.riscos)}")

---
## 9. Resumo

| Peca | Triagem | Planejamento |
|---|---|---|
| Contrato | `Triagem` | `Plano` (com `PassoPlano`) |
| O que e cada campo | `Field(description=...)` | `Field(description=...)` |
| Sistema | classificar pela politica, citar fontes | plano executavel, cobrir todos os CA, citar fontes |
| Usuario | issue + contexto | issue + contexto + **triagem** (+ feedback) |
| Chamada | `chamar(Triagem, ...)` | `chamar(Plano, ...)` |
| Conferencia sem LLM | fontes existem? prioridade coerente? | todos os CA cobertos? fontes existem? |

Tres licoes:

1. **Contrato + prompt + `with_structured_output`** - a mesma receita serve para qualquer etapa.
   O contrato diz **o que e** cada campo (`description`); o prompt de sistema diz **como decidir**.
2. **A saida de uma etapa e entrada da proxima**, serializada com `model_dump_json`.
3. **O que o prompt pede, o codigo confere.** O LLM e instruido a citar fontes e cobrir
   criterios; conjuntos simples verificam se ele cumpriu.

## 10. Exercicios

1. Troque o modelo para `claude-haiku-4-5` e rode as etapas 6 e 7 de novo. A triagem mudou?
   A cobertura dos criterios continua completa?
2. Remova o trecho `politica-de-engenharia#2` do contexto e rode a triagem. O que acontece com
   a justificativa e com as `fontes`?
3. Retire a secao `TRIAGEM JA FEITA` do `prompt_plano` e gere o plano de novo. O plano
   continua coerente com a severidade e os componentes da triagem?
4. Apague a `description` de `testes` em `Plano`, rode a celula 2.1 e depois a etapa 7.
   Como mudou o formato dos testes gerados? Escreva uma descricao pedindo nomes de funcao
   `test_...` e compare.
5. Escreva um `if` que, quando faltar criterio na secao 7.1, chame `planejar` de novo com um
   feedback listando os CA faltantes - voce tera implementado o laco de correcao do DevFlow.